In [1]:
import os
os.chdir('/home/svs25/SAE')

os.CUDA_VISIBLE_DEVICES = "1"

In [2]:
from pathlib import Path
import json
import torch
import pandas as pd
import numpy as np

REPO_ROOT = Path(".")

# Checkpoint from training
CKPT_PATH = REPO_ROOT / "results/topk_sae_layer6_d4096_k32_run1/best.pt"

# Train memmap / metadata
MEMMAP_BIN = REPO_ROOT / "data/embeddings_memmap/layer6/s1a70_train_acts.f16.bin"
MEMMAP_META = REPO_ROOT / "data/embeddings_memmap/layer6/s1a70_train_acts.meta.json"

# Raw flattened activations with sequence_ids and token_positions
RAW_PT_PATH = REPO_ROOT / "data/embeddings/s1a70_train_layer6_raw.pt"

# Processed CSV with original sequences
CSV_PATH = REPO_ROOT / "data/processed/s1a70/s1a70_train.csv"

# Analysis outputs
OUTPUT_DIR = REPO_ROOT / "results/topk_sae_layer6_d4096_k32_run1/analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Previously saved latent summaries/examples
SUMMARY_JSON = OUTPUT_DIR / "latent_summary.json"
TOP_EXAMPLES_JSON = OUTPUT_DIR / "top_examples.json"

print("Using files:")
for p in [
    CKPT_PATH, MEMMAP_BIN, MEMMAP_META, RAW_PT_PATH, CSV_PATH, SUMMARY_JSON, TOP_EXAMPLES_JSON
]:
    print(" -", p, "| exists:", p.exists())

Using files:
 - results/topk_sae_layer6_d4096_k32_run1/best.pt | exists: True
 - data/embeddings_memmap/layer6/s1a70_train_acts.f16.bin | exists: True
 - data/embeddings_memmap/layer6/s1a70_train_acts.meta.json | exists: True
 - data/embeddings/s1a70_train_layer6_raw.pt | exists: True
 - data/processed/s1a70/s1a70_train.csv | exists: True
 - results/topk_sae_layer6_d4096_k32_run1/analysis/latent_summary.json | exists: True
 - results/topk_sae_layer6_d4096_k32_run1/analysis/top_examples.json | exists: True


In [3]:
seq_df = pd.read_csv(CSV_PATH).reset_index().rename(columns={"index": "sequence_id"})
seq_df.head(3)

,sequence_id,accession,entry_name,reviewed,organism,taxon_id,sequence,length,class_label,class_name,...,has_ps00134,has_ps00135,has_both_catalytic_motifs,interpro_ids,prosite_ids,motif_class,cluster_id,cluster_rep,split,cluster_size
0,0,P08217,CEL2A_HUMAN,True,Homo sapiens (Human),9606,MIRTLLLSTLVAGALSCGDPTYPPYVTRVVGGEEARPNSWPWQVSL...,269,1,S1A_trypsin_chymotrypsin,...,True,True,True,IPR050850;IPR009003;IPR043504;IPR001314;IPR001...,PS50240;PS00134;PS00135;,both_motifs,pos__I3MWU7,I3MWU7,train,130
1,1,P08218,CEL2B_HUMAN,True,Homo sapiens (Human),9606,MIRTLLLSTLVAGALSCGVSTYAPDMSRMLGGEEARPNSWPWQVSL...,269,1,S1A_trypsin_chymotrypsin,...,True,True,True,IPR050850;IPR009003;IPR043504;IPR001314;IPR001...,PS50240;PS00134;PS00135;,both_motifs,pos__I3MWU7,I3MWU7,train,130
2,2,P08246,ELNE_HUMAN,True,Homo sapiens (Human),9606,MTLGRRLACLFLACVLPALLLGGTALASEIVGGRRARPHAWPFMVS...,267,1,S1A_trypsin_chymotrypsin,...,True,True,True,IPR050850;IPR009003;IPR043504;IPR001314;IPR001...,PS50240;PS00134;PS00135;,both_motifs,pos__P08246,P08246,train,13


In [4]:
with open(SUMMARY_JSON, "r") as f:
    latent_summary = json.load(f)

with open(TOP_EXAMPLES_JSON, "r") as f:
    top_examples_output = json.load(f)

summary_df = pd.DataFrame(latent_summary)
summary_df.head()

,latent_idx,fire_count,fire_fraction,mean_active_value,max_value
0,2491,2146352,0.208966,6.223986,11.579073
1,3034,1633556,0.159041,1.015022,6.446189
2,474,1573335,0.153178,2.079281,7.984866
3,3691,1391219,0.135447,0.981014,5.741388
4,24,1318787,0.128395,0.919013,6.838675


In [5]:
raw_obj = torch.load(RAW_PT_PATH, map_location="cpu")

print(type(raw_obj))
if isinstance(raw_obj, dict):
    print(raw_obj.keys())

sequence_ids = raw_obj["sequence_ids"]
token_positions = raw_obj["token_positions"]

print(sequence_ids.shape, token_positions.shape)
print(sequence_ids[:10], token_positions[:10])

/tmp/ipykernel_3583408/2605975550.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  raw_obj = torch.load(RAW_PT_PATH, map_location="cpu")


<class 'dict'>
dict_keys(['activations', 'sequence_ids', 'token_positions', 'lengths', 'split_name', 'source_csv', 'sequence_column', 'model_name', 'hidden_index', 'save_dtype'])
torch.Size([10271296]) torch.Size([10271296])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0]) tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])


In [6]:
label_cols = [
    "sequence_id",
    "accession",
    "entry_name",
    "class_label",
    "class_name",
    "has_ipr001314",
    "has_ps00134",
    "has_ps00135",
    "has_both_catalytic_motifs",
    "motif_class",
]

seq_labels_df = seq_df[label_cols].copy()
seq_labels_df["is_s1a"] = (seq_labels_df["class_label"] == 1).astype(int)
seq_labels_df["has_trypsin_domain"] = seq_labels_df["has_ipr001314"].astype(int)
seq_labels_df["has_both_motifs"] = seq_labels_df["has_both_catalytic_motifs"].astype(int)

seq_labels_df.head()

,sequence_id,accession,entry_name,class_label,class_name,has_ipr001314,has_ps00134,has_ps00135,has_both_catalytic_motifs,motif_class,is_s1a,has_trypsin_domain,has_both_motifs
0,0,P08217,CEL2A_HUMAN,1,S1A_trypsin_chymotrypsin,True,True,True,True,both_motifs,1,1,1
1,1,P08218,CEL2B_HUMAN,1,S1A_trypsin_chymotrypsin,True,True,True,True,both_motifs,1,1,1
2,2,P08246,ELNE_HUMAN,1,S1A_trypsin_chymotrypsin,True,True,True,True,both_motifs,1,1,1
3,3,P23946,CMA1_HUMAN,1,S1A_trypsin_chymotrypsin,True,True,True,True,both_motifs,1,1,1
4,4,P49862,KLK7_HUMAN,1,S1A_trypsin_chymotrypsin,True,True,True,True,both_motifs,1,1,1


In [7]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

LATENT = 3228
DEVICE = "cpu"

# Load raw flattened activation metadata
raw_obj = torch.load(RAW_PT_PATH, map_location="cpu")
print("raw_obj keys:", raw_obj.keys())

sequence_ids = raw_obj["sequence_ids"]

# Rebuild memmap dataset
from topk_sae.dataset.memmap_dataset import ActivationMemmap

acts = ActivationMemmap(
    str(MEMMAP_BIN),
    str(MEMMAP_META),
)

print("len(acts):", len(acts))
print("sequence_ids shape:", sequence_ids.shape)

# Load model correctly
from topk_sae.models.train_topk_sae import TopKSAE

def load_model_from_checkpoint(ckpt_path: Path, device: str = "cpu"):
    ckpt = torch.load(ckpt_path, map_location="cpu")

    config = ckpt["config"]
    model = TopKSAE(
        d_in=config["d_in"],
        d_sae=config["d_sae"],
        k=config["k"],
        use_pre_bias=True,
        use_post_bias=True,
        normalize_decoder=True,
    )
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(device)
    model.eval()

    input_mean = ckpt.get("input_mean", None)
    input_std = ckpt.get("input_std", None)

    if input_mean is not None:
        input_mean = input_mean.to(device)
    if input_std is not None:
        input_std = input_std.to(device)

    return model, config, ckpt, input_mean, input_std

sae, cfg, ckpt, input_mean, input_std = load_model_from_checkpoint(CKPT_PATH, device=DEVICE)
print(cfg)
print("Has input_mean:", input_mean is not None)
print("Has input_std:", input_std is not None)

/tmp/ipykernel_3583408/4144832469.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  raw_obj = torch.load(RAW_PT_PATH, map_location="cpu")


raw_obj keys: dict_keys(['activations', 'sequence_ids', 'token_positions', 'lengths', 'split_name', 'source_csv', 'sequence_column', 'model_name', 'hidden_index', 'save_dtype'])
len(acts): 10271296
sequence_ids shape: torch.Size([10271296])
{'d_in': 384, 'd_sae': 4096, 'k': 32}
Has input_mean: True
Has input_std: True


/tmp/ipykernel_3583408/4144832469.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


In [8]:
print(type(acts))
print(dir(acts))

<class 'topk_sae.dataset.memmap_dataset.ActivationMemmap'>
['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_arr', 'bin_path', 'dim', 'dtype', 'get_rows', 'meta', 'meta_path', 'sample_torch_batch', 'shape']


In [9]:
chunk_size = 8192
latent_vals = []

with torch.no_grad():
    for start in range(0, len(acts), chunk_size):
        end = min(start + chunk_size, len(acts))

        idx = np.arange(start, end, dtype=np.int64)
        x = acts.get_rows(idx)

        if not torch.is_tensor(x):
            x = torch.from_numpy(x)

        x = x.to(DEVICE).float()   # <- important fix

        if input_mean is not None and input_std is not None:
            x = (x - input_mean) / input_std

        out = sae(x)
        
        z_chunk = out["z"][:, LATENT].cpu()
        latent_vals.append(z_chunk)

z_latent = torch.cat(latent_vals, dim=0)

print("z_latent shape:", z_latent.shape)
print("sequence_ids shape:", sequence_ids.shape)
assert len(z_latent) == len(sequence_ids)

z_latent shape: torch.Size([10271296])
sequence_ids shape: torch.Size([10271296])


In [10]:
seq_ids_np = sequence_ids.cpu().numpy() if torch.is_tensor(sequence_ids) else np.asarray(sequence_ids)
z_np = z_latent.cpu().numpy() if torch.is_tensor(z_latent) else np.asarray(z_latent)

assert len(seq_ids_np) == len(z_np)

token_df = pd.DataFrame({
    "sequence_id": seq_ids_np,
    "activation": z_np,
    "is_active": z_np > 0,
})

seq_latent_df = token_df.groupby("sequence_id").agg(
    any_active=("is_active", "max"),
    fire_count=("is_active", "sum"),
    max_activation=("activation", "max"),
    mean_activation_all_tokens=("activation", "mean"),
    activation_sum=("activation", "sum"),
).reset_index()

seq_latent_df["mean_activation_active_only"] = 0.0
mask = seq_latent_df["fire_count"] > 0
seq_latent_df.loc[mask, "mean_activation_active_only"] = (
    seq_latent_df.loc[mask, "activation_sum"] / seq_latent_df.loc[mask, "fire_count"]
)

latent_enrichment_df = seq_latent_df.merge(
    seq_labels_df,
    on="sequence_id",
    how="left"
)

latent_enrichment_df.sort_values("max_activation", ascending=False, inplace=True)
latent_enrichment_df.head(20)

,sequence_id,any_active,fire_count,max_activation,mean_activation_all_tokens,activation_sum,mean_activation_active_only,accession,entry_name,class_label,class_name,has_ipr001314,has_ps00134,has_ps00135,has_both_catalytic_motifs,motif_class,is_s1a,has_trypsin_domain,has_both_motifs
17924,17924,True,8,68.831558,0.389654,94.685898,11.835737,Q2FXC8,SPLF_STAA8,0,background,False,False,False,False,neither_motif,0,0,0
29514,29514,True,3,65.258385,0.326390,93.347496,31.115832,P0C0Q1,GSEA_STAEP,0,background,False,False,False,False,neither_motif,0,0,0
11964,11964,True,10,60.050537,0.295997,85.247025,8.524702,A0A670ZN55,A0A670ZN55_PSETE,1,S1A_trypsin_chymotrypsin,True,False,False,False,neither_motif,1,1,0
1150,1150,True,3,59.715805,0.319807,77.712982,25.904327,A0A345XT64,A0A345XT64_9ACTN,1,S1A_trypsin_chymotrypsin,True,False,False,False,neither_motif,1,1,0
11246,11246,True,7,59.715332,0.315550,77.625336,11.089334,A0A0J9R7V3,A0A0J9R7V3_DROSI,1,S1A_trypsin_chymotrypsin,True,False,False,False,neither_motif,1,1,0
9015,9015,True,3,59.313454,0.309773,75.894333,25.298111,A0A345XU08,A0A345XU08_9ACTN,1,S1A_trypsin_chymotrypsin,True,False,False,False,neither_motif,1,1,0
6867,6867,True,7,59.168728,0.284083,83.804535,11.972076,A0A670ZMP3,A0A670ZMP3_PSETE,1,S1A_trypsin_chymotrypsin,True,False,False,False,neither_motif,1,1,0
12990,12990,True,7,59.162521,0.279222,83.208244,11.886892,A0A8X7XC21,A0A8X7XC21_POLSE,1,S1A_trypsin_chymotrypsin,True,False,True,False,serine_only,1,1,0
3128,3128,True,7,59.137360,0.283963,77.805801,11.115114,A0ABD2GS77,A0ABD2GS77_PAGBO,1,S1A_trypsin_chymotrypsin,True,False,True,False,serine_only,1,1,0
1634,1634,True,7,58.915215,0.264090,68.927399,9.846771,A0A671NGA0,A0A671NGA0_9TELE,1,S1A_trypsin_chymotrypsin,True,False,True,False,serine_only,1,1,0


In [11]:
summary = latent_enrichment_df.groupby("is_s1a").agg(
    n_sequences=("sequence_id", "count"),
    hit_rate=("any_active", "mean"),
    mean_fire_count=("fire_count", "mean"),
    mean_max_activation=("max_activation", "mean"),
    median_max_activation=("max_activation", "median"),
).reset_index()

summary

,is_s1a,n_sequences,hit_rate,mean_fire_count,mean_max_activation,median_max_activation
0,0,24732,0.325732,0.516497,0.400054,0.00000
1,1,13586,0.993081,3.073900,45.205776,47.21032


In [13]:
from sklearn.metrics import roc_auc_score
from scipy.stats import mannwhitneyu

auroc_max_activation = roc_auc_score(
    latent_enrichment_df["is_s1a"],
    latent_enrichment_df["max_activation"]
)

pos = latent_enrichment_df.loc[latent_enrichment_df["is_s1a"] == 1, "max_activation"]
neg = latent_enrichment_df.loc[latent_enrichment_df["is_s1a"] == 0, "max_activation"]

u_stat, p_value = mannwhitneyu(pos, neg, alternative="greater")

hit_rates = latent_enrichment_df.groupby("is_s1a")["any_active"].mean()

print("AUROC (max_activation):", auroc)
print("Mann-Whitney p-value:", p_value)
print("Hit rate S1A:", hit_rates.loc[1])
print("Hit rate background:", hit_rates.loc[0])
print("Fold-enrichment:", hit_rates.loc[1] / hit_rates.loc[0])

AUROC (max_activation): 0.9910946450021962
Mann-Whitney p-value: 0.0
Hit rate S1A: 0.9930811129103488
Hit rate background: 0.3257318453825004
Fold-enrichment: 3.0487688784134495


In [14]:
auroc_fire_count = roc_auc_score(
    latent_enrichment_df["is_s1a"],
    latent_enrichment_df["fire_count"]
)

pos = latent_enrichment_df.loc[latent_enrichment_df["is_s1a"] == 1, "fire_count"]
neg = latent_enrichment_df.loc[latent_enrichment_df["is_s1a"] == 0, "fire_count"]

u_stat, p_value = mannwhitneyu(pos, neg, alternative="greater")

hit_rates = latent_enrichment_df.groupby("is_s1a")["any_active"].mean()

print("AUROC (fire_count):", auroc_fire_count)
print("Mann-Whitney p-value:", p_value)

AUROC (fire_count): 0.9438434649205416
Mann-Whitney p-value: 0.0
